# Evaluation and Deployment

This notebook evaluates the performance of the content-based homestay recommendation system and prepares the recommendation engine for deployment.

The notebook performs the following tasks:

1. Defines recommendation relevance criteria.
2. Evaluates recommendation quality using Precision@K.
3. Evaluates recommendation quality using Recall@K.
4. Evaluates ranking performance using NDCG@K.
5. Computes overall recommendation performance.
6. Saves trained recommendation components for deployment.
7. Exports files required for the Streamlit application.

The evaluation process helps measure how effectively the recommendation system identifies relevant homestays based on shared location characteristics, category, and amenities.

In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.metrics import ndcg_score

import joblib

In [2]:
# ==========================================
# LOAD PREPARED DATASET
# ==========================================

df = pd.read_csv(
    "../data/final/homestays_prepared.csv"
)

print(
    f"Dataset Shape: {df.shape}"
)

df.head()

Dataset Shape: (1157, 39)


,homestay_id,homestay_name,owner_name,category,district,block,village,owner_email,owner_mobile,google_name,...,breakfast,mountain_view,room_service,bonfire_barbeque,pickup_dropoff_service,description,price_band,amenity_text,location_text,feature_text
0,1,Revere Homestay,Mr. Riwaj Pradhan,Silver,Kalimpong,Municipality,"8Th Mile, Kalimpong",riwajpradhan10@gmail.com,9800686780,Revere Homsetay,...,1,0,0,0,0,"Revere Homestay in 8th Mile, Kalimpong Village...",mid_range,parking breakfast,"Kalimpong Municipality 8Th Mile, Kalimpong","Revere Homestay in 8th Mile, Kalimpong Village..."
1,2,Mansarover Homestay,Miss Tina Mani Gurung,Gold,Kalimpong,Municipality,"Chandralok, Kalimpong",santabgurung53@gmail.com,9932234895,Mansarover Homestay / Flora & Transport,...,0,0,0,1,0,"Mansarover Homestay in Chandralok Village, Mun...",premium,wifi parking bonfire_barbeque,"Kalimpong Municipality Chandralok, Kalimpong","Mansarover Homestay in Chandralok Village, Mun..."
2,3,Bethany Homestay,Anupama Tamang,Silver,Kalimpong,Kalimpong I,Dr.Grahams Home Block B,wangchuck20199@gmial.com,8348993048,Bethany Homestay Kalimpong,...,1,1,1,1,0,BETHANY HOMESTAY is positioned within Kalimpon...,budget,parking breakfast mountain_view room_service b...,Kalimpong Kalimpong I Dr.Grahams Home Block B,BETHANY HOMESTAY is positioned within Kalimpon...
3,4,S3 Homestay,Sangita Rai,Silver,Kalimpong,Kalimpong I,Upper Echhey Dara Gaon Kalimpong,sangitasankalp@gmail.com,9933410313,Sunrise Inn Homestay,...,1,1,1,1,0,"S3 Homestay in Upper Echhey Dara Gaon, Kalimpo...",mid_range,parking breakfast mountain_view room_service b...,Kalimpong Kalimpong I Upper Echhey Dara Gaon K...,"S3 Homestay in Upper Echhey Dara Gaon, Kalimpo..."
4,5,Bajarangi Homestay,Kamal Kumar Sharma,Silver,Kalimpong,Kalimpong I,Singi Samalbong Kalimpong,bajrangihomestay@gmail.com,8670450557,Bajrangi Homestay,...,1,1,0,1,0,BAJARANGI HOMESTAY in SINGI SAMALBONG KALIMPON...,mid_range,wifi parking breakfast mountain_view bonfire_b...,Kalimpong Kalimpong I Singi Samalbong Kalimpong,BAJARANGI HOMESTAY in SINGI SAMALBONG KALIMPON...


In [3]:
# ==========================================
# RELEVANCE FUNCTION
# ==========================================

def is_relevant(
    source_row,
    target_row
):

    same_block = (
        source_row["block"] ==
        target_row["block"]
    )

    same_category = (
        source_row["category"] ==
        target_row["category"]
    )

    shared_amenities = sum([

        source_row["wifi"] == 1 and
        target_row["wifi"] == 1,

        source_row["parking"] == 1 and
        target_row["parking"] == 1,

        source_row["breakfast"] == 1 and
        target_row["breakfast"] == 1,

        source_row["mountain_view"] == 1 and
        target_row["mountain_view"] == 1,

        source_row["room_service"] == 1 and
        target_row["room_service"] == 1,

        source_row["bonfire_barbeque"] == 1 and
        target_row["bonfire_barbeque"] == 1,

        source_row["pickup_dropoff_service"] == 1 and
        target_row["pickup_dropoff_service"] == 1

    ])

    return (
        same_block and
        same_category and
        shared_amenities >= 3
    )

In [4]:
# ==========================================
# PRECISION@K
# ==========================================

def precision_at_k(
    source_index,
    recommended_indices,
    k=5
):

    source_row = df.iloc[source_index]

    recommendations = (
        recommended_indices[:k]
    )

    relevant_count = 0

    for idx in recommendations:

        target_row = df.iloc[idx]

        if is_relevant(
            source_row,
            target_row
        ):
            relevant_count += 1

    return (
        relevant_count / k
    )

In [5]:
# ==========================================
# RECALL@K
# ==========================================

def recall_at_k(
    source_index,
    recommended_indices,
    k=5
):

    source_row = df.iloc[source_index]

    all_relevant = []

    for idx in df.index:

        if idx == source_index:
            continue

        if is_relevant(
            source_row,
            df.iloc[idx]
        ):
            all_relevant.append(idx)

    total_relevant = len(all_relevant)

    if total_relevant == 0:
        return 0

    recommendations = (
        recommended_indices[:k]
    )

    retrieved_relevant = sum(

        1

        for idx in recommendations

        if idx in all_relevant

    )

    return (
        retrieved_relevant /
        total_relevant
    )

In [6]:
# ==========================================
# NDCG@K
# ==========================================

def ndcg_at_k(
    source_index,
    recommended_indices,
    k=5
):

    source_row = df.iloc[source_index]

    y_true = []
    y_score = []

    for rank, idx in enumerate(
        recommended_indices[:k]
    ):

        target_row = df.iloc[idx]

        relevance = int(

            is_relevant(
                source_row,
                target_row
            )

        )

        y_true.append(
            relevance
        )

        y_score.append(
            k - rank
        )

    return ndcg_score(
        [y_true],
        [y_score]
    )

In [7]:
# ==========================================
# LOAD SAVED COMPONENTS
# ==========================================

similarity_matrix = joblib.load(
    "../models/cosine_similarity.pkl"
)

In [8]:
# ==========================================
# RECOMMEND HOMESTAYS
# ==========================================

def recommend_homestays(
    homestay_index,
    top_n=5
):

    similarity_scores = list(
        enumerate(
            similarity_matrix[
                homestay_index
            ]
        )
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = (
        similarity_scores[1:top_n+1]
    )

    indices = [

        item[0]

        for item in similarity_scores

    ]

    return indices

In [9]:
# ==========================================
# MODEL EVALUATION
# ==========================================

precision_scores = []
recall_scores = []
ndcg_scores = []

for idx in range(len(df)):

    recommendations = (
        recommend_homestays(
            idx,
            top_n=5
        )
    )

    precision_scores.append(

        precision_at_k(
            idx,
            recommendations,
            k=5
        )

    )

    recall_scores.append(

        recall_at_k(
            idx,
            recommendations,
            k=5
        )

    )

    ndcg_scores.append(

        ndcg_at_k(
            idx,
            recommendations,
            k=5
        )

    )

In [10]:
# ==========================================
# FINAL RESULTS
# ==========================================

print(
    f"Precision@5 : {np.mean(precision_scores):.4f}"
)

print(
    f"Recall@5    : {np.mean(recall_scores):.4f}"
)

print(
    f"NDCG@5      : {np.mean(ndcg_scores):.4f}"
)

Precision@5 : 0.4029
Recall@5    : 0.0237
NDCG@5      : 0.6007


In [11]:
relevant_counts = []

for idx in range(len(df)):

    source_row = df.iloc[idx]

    count = 0

    for j in range(len(df)):

        if idx == j:
            continue

        if is_relevant(
            source_row,
            df.iloc[j]
        ):
            count += 1

    relevant_counts.append(count)

print(
    pd.Series(relevant_counts).describe()
)

count    1157.000000
mean      142.456353
std       118.568301
min         0.000000
25%        35.000000
50%       132.000000
75%       207.000000
max       428.000000
dtype: float64
